# Phase 2, Item 4: convert both models to ONNX int8

Checklist item: *"Convert both models to ONNX int8 -- Done when p95 retrieval
latency is under 1.5s on the free Space and recall is within 1 point of
float32."* "The free Space" = a Hugging Face Spaces free-tier deployment,
which is **CPU-only** -- so this notebook benchmarks CPU latency, not GPU
latency, even though it runs on a GPU instance for convenience during export.

**Scope decision, made deliberately, not by accident:** `embed_text()`
returns two things from one BGE-M3 model call -- a dense vector (meaning
search) and a sparse vector (exact-word search), and both are used live on
every query (`dense_search` and `sparse_search` both depend on it). The dense
output is a standard CLS-token + L2-normalize pooling over the base
transformer's last hidden state, cleanly exportable with standard tooling.
The sparse output is a *custom* extra linear head bolted onto BGE-M3 that
standard ONNX export tools do not know how to convert -- attempting it blind,
without a way to test locally first, risks silently corrupting sparse search
(which is currently pulling real weight: 0.542 Recall@1 alone, actually
*better* than dense alone at 0.462) with no error to warn us it broke.

**So: this notebook converts and quantizes the dense embedding output and the
reranker. Sparse search stays on the original float32 model, unchanged.**
That is what "both models" means here -- the embedding model and the
reranker, the two components the rest of this project's docs already refer
to as "the two models" (see `retrieval/pipeline.py`'s warmup() docstring).

**Runtime -> Change runtime type -> T4 GPU** (needed to load the original
fp16 models for comparison; the actual latency benchmark below forces CPU).

In [1]:
# diffusers is unrelated to what this notebook does (text-only ONNX export, no image/diffusion models), but optimum probes for it on import -- Kaggle's base image ships a diffusers/huggingface_hub version pairing that crashes that probe instead of failing gracefully, taking ORTModelForSequenceClassification down with it. Uninstalling it first sidesteps the broken import entirely.
!pip uninstall -y diffusers -q
!pip install -q "optimum[onnxruntime]" onnx FlagEmbedding qdrant-client psutil


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 15.8 MB/s eta 0:00:00


In [2]:
import os

work_dir = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
%cd $work_dir
!rm -rf naari-ai
!git clone --branch sana/phase2-service --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

print("cloned OK")

/kaggle/working
Cloning into 'naari-ai'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 109 (delta 5), reused 80 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 906.99 KiB | 5.12 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/kaggle/working/naari-ai
cloned OK


In [3]:
import os


def _get_qdrant_creds():
    """Kaggle: Add-ons -> Secrets, labelled exactly QDRANT_URL and QDRANT_API_KEY
    (both must be attached to this notebook). Anywhere else: returns None so the
    caller falls back to typing them in."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return None
    try:
        secrets = UserSecretsClient()
        return secrets.get_secret("QDRANT_URL"), secrets.get_secret("QDRANT_API_KEY")
    except Exception as exc:
        print(f"Kaggle Secrets lookup failed: {exc}")
        print("Check Add-ons -> Secrets: the labels must be exactly QDRANT_URL and "
              "QDRANT_API_KEY, and both must be attached to this notebook "
              "(the toggle next to each one).")
        return None


_creds = _get_qdrant_creds()
if _creds is None:
    from getpass import getpass

    _creds = (getpass("Qdrant cluster URL: "), getpass("Qdrant API key: "))
    print("using manually entered credentials")
else:
    print("loaded Qdrant credentials from Kaggle Secrets")

# pipeline.py reads these from os.environ directly (12-factor style) --
# set them as real env vars, not just local Python variables.
os.environ["QDRANT_URL"], os.environ["QDRANT_API_KEY"] = _creds
print("env vars set")


loaded Qdrant credentials from Kaggle Secrets
env vars set


In [4]:
import csv

with open("eval/gold_eval_280_linked.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

for row in gold_rows:
    raw = row.get("acceptable_answer_ids", "").strip()
    row["_acceptable_ids"] = set(raw.split(";")) if raw else {row["correct_answer_id"]}

print(f"{len(gold_rows)} gold rows loaded")
assert len(gold_rows) == 248

import gc
import psutil


def print_mem(label=""):
    rss_gb = psutil.Process().memory_info().rss / (1024 ** 3)
    msg = f"[mem] {label}: {rss_gb:.2f}GB RSS"
    try:
        import torch
        if torch.cuda.is_available():
            msg += f", {torch.cuda.memory_allocated()/(1024**3):.2f}GB GPU"
    except ImportError:
        pass
    print(msg)


print_mem("startup")

248 gold rows loaded
[mem] startup: 0.10GB RSS, 0.00GB GPU


## Step 1: export and quantize the reranker (`bge-reranker-v2-m3`)

This is a standard sequence-classification cross-encoder -- one logit per
(query, passage) pair, sigmoid-normalized. `optimum`'s auto-export handles
this architecture directly, no custom pooling needed.

In [5]:
import os

# outside the naari-ai/ clone directory on purpose -- restarting the kernel
# and re-running from the top re-clones the repo (`rm -rf naari-ai`), which
# would otherwise wipe out everything exported so far.
_work_dir = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
RERANKER_ID = "BAAI/bge-reranker-v2-m3"
rerank_fp32_dir = f"{_work_dir}/onnx_models/reranker_fp32"
rerank_int8_dir = f"{_work_dir}/onnx_models/reranker_int8"

if os.path.exists(f"{rerank_int8_dir}/model_quantized.onnx"):
    print(f"reranker already exported+quantized at {rerank_int8_dir} -- skipping "
          f"(delete that folder to force a redo)")
else:
    from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    from transformers import AutoTokenizer

    tokenizer_rerank = AutoTokenizer.from_pretrained(RERANKER_ID)
    ort_reranker_fp32 = ORTModelForSequenceClassification.from_pretrained(RERANKER_ID, export=True)
    ort_reranker_fp32.save_pretrained(rerank_fp32_dir)
    tokenizer_rerank.save_pretrained(rerank_fp32_dir)
    print("exported reranker to ONNX (fp32)")

    quantizer = ORTQuantizer.from_pretrained(rerank_fp32_dir)
    # avx2 (not avx512_vnni) deliberately -- broadly compatible with whatever CPU
    # the free Space actually runs on, not just this notebook's own hardware.
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=rerank_int8_dir, quantization_config=qconfig)
    tokenizer_rerank.save_pretrained(rerank_int8_dir)
    print("quantized reranker to int8")

    # the fp32 export object is ~2.2GB and everything from here on reads from
    # the saved-to-disk int8 version instead -- free it now, not at the end.
    del ort_reranker_fp32, quantizer
    gc.collect()
    print_mem("after reranker export+quantize, fp32 export object freed")


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


exported reranker to ONNX (fp32)


The tokenizer you are loading from '/kaggle/working/onnx_models/reranker_fp32' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/kaggle/working/onnx_models/reranker_fp32' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


quantized reranker to int8
[mem] after reranker export+quantize, fp32 export object freed: 4.43GB RSS, 0.00GB GPU


In [6]:
import torch

ort_reranker_int8 = ORTModelForSequenceClassification.from_pretrained(
    rerank_int8_dir, file_name="model_quantized.onnx", provider="CPUExecutionProvider"
)
tokenizer_rerank_int8 = AutoTokenizer.from_pretrained(rerank_int8_dir)


def onnx_rerank_score(query, passage):
    inputs = tokenizer_rerank_int8([[query, passage]], padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        logits = ort_reranker_int8(**inputs).logits.view(-1).float()
    return torch.sigmoid(logits).item()


print("int8 reranker loaded on CPU, ready")

The tokenizer you are loading from '/kaggle/working/onnx_models/reranker_int8' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


int8 reranker loaded on CPU, ready


## Step 2: sanity-check the reranker BEFORE doing anything else

Compares the original float32 model against the new int8 one on one clearly
relevant pair and one clearly irrelevant pair. If these diverge, **stop here**
-- something in the export or quantization is wrong, and the full benchmark
below would just waste an hour producing garbage numbers on top of it.

In [7]:
import numpy as np
from FlagEmbedding import FlagReranker
import retrieval.rerank as rerank_module

original_reranker = FlagReranker(RERANKER_ID, use_fp16=True)
# register this as retrieval.rerank's own singleton so the recall benchmark's
# rerank() call later reuses THIS instance instead of loading a second ~2GB+
# copy of the exact same model.
rerank_module._model = original_reranker
print_mem("after loading the original fp16 reranker (shared with later cells)")


def _as_float(score):
    """compute_score() returns a plain float on some FlagEmbedding versions and
    a numpy array on others -- normalise to a float either way, the same thing
    retrieval/rerank.py does before storing a rerank_score."""
    return float(np.asarray(score, dtype=float).reshape(-1)[0])


relevant_pair = (gold_rows[0]["query"], gold_rows[0]["query"])  # a query against itself: should score very high
irrelevant_pair = (gold_rows[0]["query"], gold_rows[100]["query"])  # two unrelated queries: should score low

for label, (q, p) in [("relevant (self-pair)", relevant_pair), ("irrelevant (unrelated pair)", irrelevant_pair)]:
    orig = _as_float(original_reranker.compute_score([[q, p]], normalize=True))
    new = _as_float(onnx_rerank_score(q, p))
    print(f"{label}: original={orig:.4f}  onnx-int8={new:.4f}")

print("\nExpect: relevant pair high on both (>0.8ish), irrelevant pair low on both (<0.2ish), "
      "and the two numbers roughly agreeing with each other. If they don't, stop and investigate "
      "before running the full benchmark below.")


[mem] after loading the original fp16 reranker (shared with later cells): 5.41GB RSS, 0.00GB GPU


Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.78s/it]


relevant (self-pair): original=1.0000  onnx-int8=1.0000


Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]


irrelevant (unrelated pair): original=0.0000  onnx-int8=0.0000

Expect: relevant pair high on both (>0.8ish), irrelevant pair low on both (<0.2ish), and the two numbers roughly agreeing with each other. If they don't, stop and investigate before running the full benchmark below.


## Checkpoint: safe to restart the kernel here

The reranker is exported, quantized, and saved to disk outside the repo
clone -- restarting the kernel now does not lose it. If memory is already
tight, **restart the kernel now**, then re-run from the top:
- The install + clone cells are cheap to repeat.
- The `gold_rows` load and `print_mem` helper cell is cheap to repeat.
- The reranker export cell will print "already exported... skipping" and
  move on in a couple seconds instead of redoing the heavy work.
- Continue from Step 3 (the embedder) below.

If memory looks fine so far, just keep going -- this checkpoint is a safety
net, not a required stop.

## Step 3: export and quantize the embedder's dense output (`bge-m3`)

BGE-M3's dense vector is the base transformer's last-hidden-state CLS token
(position 0), L2-normalized -- no extra learned head, unlike the sparse
branch. `ORTModelForFeatureExtraction` gives access to the raw last hidden
state; the CLS-pool + normalize step is done manually below to exactly match
what `FlagEmbedding`'s `BGEM3FlagModel` does internally for the dense branch.

In [8]:
import os
from optimum.onnxruntime import ORTModelForFeatureExtraction

_work_dir = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
EMBED_ID = "BAAI/bge-m3"
embed_fp32_dir = f"{_work_dir}/onnx_models/embed_fp32"
embed_int8_dir = f"{_work_dir}/onnx_models/embed_int8"

if os.path.exists(f"{embed_int8_dir}/model_quantized.onnx"):
    print(f"embedder already exported+quantized at {embed_int8_dir} -- skipping "
          f"(delete that folder to force a redo)")
else:
    from optimum.onnxruntime import ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    from transformers import AutoTokenizer

    tokenizer_embed = AutoTokenizer.from_pretrained(EMBED_ID)
    ort_embed_fp32 = ORTModelForFeatureExtraction.from_pretrained(EMBED_ID, export=True)
    ort_embed_fp32.save_pretrained(embed_fp32_dir)
    tokenizer_embed.save_pretrained(embed_fp32_dir)
    print("exported embedder to ONNX (fp32)")

    quantizer = ORTQuantizer.from_pretrained(embed_fp32_dir)
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=embed_int8_dir, quantization_config=qconfig)
    tokenizer_embed.save_pretrained(embed_int8_dir)
    print("quantized embedder to int8")

    del ort_embed_fp32, quantizer
    gc.collect()
    print_mem("after embedder export+quantize, fp32 export object freed")


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

The model BAAI/bge-m3 was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`


...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


exported embedder to ONNX (fp32)


The tokenizer you are loading from '/kaggle/working/onnx_models/embed_fp32' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/kaggle/working/onnx_models/embed_fp32' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


quantized embedder to int8
[mem] after embedder export+quantize, fp32 export object freed: 8.13GB RSS, 0.00GB GPU


In [9]:
import torch.nn.functional as F
import numpy as np

ort_embed_int8 = ORTModelForFeatureExtraction.from_pretrained(
    embed_int8_dir, file_name="model_quantized.onnx", provider="CPUExecutionProvider"
)
tokenizer_embed_int8 = AutoTokenizer.from_pretrained(embed_int8_dir)


def onnx_dense_embed(text):
    inputs = tokenizer_embed_int8([text], padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        last_hidden = ort_embed_int8(**inputs).last_hidden_state
    cls = last_hidden[:, 0, :]
    return F.normalize(cls, p=2, dim=1)[0].numpy()


print("int8 embedder loaded on CPU, ready")

The tokenizer you are loading from '/kaggle/working/onnx_models/embed_int8' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


int8 embedder loaded on CPU, ready


## Step 4: sanity-check the dense embedding -- the critical checkpoint

This is the assumption most likely to be wrong (CLS-pooling matching
`FlagEmbedding`'s internal convention). Cosine similarity between the
original and the ONNX-int8 dense vector for the same real query should be
very close to 1.0. **The assert below stops the notebook if it isn't** --
do not comment it out to "see what happens next."

In [10]:
from retrieval.embed import embed_text as original_embed_text

test_query = gold_rows[0]["query"]
orig_vec = np.array(original_embed_text(test_query)["dense"])
new_vec = onnx_dense_embed(test_query)

cos_sim = float(np.dot(orig_vec, new_vec) / (np.linalg.norm(orig_vec) * np.linalg.norm(new_vec)))
print(f"cosine similarity, original vs onnx-int8 dense embedding: {cos_sim:.4f}  (expect > 0.98)")

assert cos_sim > 0.95, (
    "ONNX dense embedding diverges too much from the original -- STOP. "
    "Do not proceed to the full benchmark below until this is fixed."
)
print("sanity check passed -- proceeding to the full benchmark")
print_mem("after dense-embedding sanity check")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

long.jpg:   0%|          | 0.00/485k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

...

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

cosine similarity, original vs onnx-int8 dense embedding: 0.9839  (expect > 0.98)
sanity check passed -- proceeding to the full benchmark
[mem] after dense-embedding sanity check: 11.43GB RSS, 0.00GB GPU


## Checkpoint: safe to restart the kernel here too

Same deal -- both models are exported and saved to disk now. The benchmark
cells below (Steps 5-6) are the heaviest section left (248 gold queries
through two full pipelines). **If memory is at all tight, restart the
kernel now** and re-run from the top -- both export cells will skip
straight past the heavy work and you will reach the benchmarks with a
clean, low-memory process instead of one that has been accumulating
allocations since the very first cell. Don't forget the credentials
cell (getpass for Qdrant) -- env vars reset on a kernel restart too,
and the recall benchmark right after this needs them.

## Step 5: latency benchmark, on CPU (the actual deployment target)

Both the float32 baseline and the int8 candidate are measured on CPU here,
deliberately -- this notebook has a GPU available, but the free Space does
not, so a GPU-vs-CPU comparison would measure the wrong thing entirely.

In [11]:
import time
from FlagEmbedding import BGEM3FlagModel

# Reload the embedder in true float32 on CPU specifically -- the checklist's
# own baseline is "float32", and it needs to run on the same hardware class
# (CPU) as the int8 candidate for the comparison to mean anything.
cpu_embed_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False, device="cpu")
cpu_rerank_model = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=False, device="cpu")

sample_queries = [row["query"] for row in gold_rows[:60]]


def time_calls(fn, items):
    latencies = []
    for item in items:
        t0 = time.perf_counter()
        fn(item)
        latencies.append((time.perf_counter() - t0) * 1000)
    return latencies


def pctl(latencies, pct):
    s = sorted(latencies)
    return s[int(pct * len(s))]


fp32_embed_lat = time_calls(
    lambda q: cpu_embed_model.encode([q], return_dense=True, return_sparse=False, return_colbert_vecs=False),
    sample_queries,
)
int8_embed_lat = time_calls(onnx_dense_embed, sample_queries)

print(f"dense embed  fp32-CPU: p50={pctl(fp32_embed_lat,0.5):.0f}ms  p95={pctl(fp32_embed_lat,0.95):.0f}ms")
print(f"dense embed  int8-CPU: p50={pctl(int8_embed_lat,0.5):.0f}ms  p95={pctl(int8_embed_lat,0.95):.0f}ms")

sample_pairs = [(gold_rows[i]["query"], gold_rows[(i + 7) % len(gold_rows)]["query"]) for i in range(60)]

fp32_rerank_lat = time_calls(lambda qp: cpu_rerank_model.compute_score([[qp[0], qp[1]]], normalize=True), sample_pairs)
int8_rerank_lat = time_calls(lambda qp: onnx_rerank_score(qp[0], qp[1]), sample_pairs)

print(f"rerank       fp32-CPU: p50={pctl(fp32_rerank_lat,0.5):.0f}ms  p95={pctl(fp32_rerank_lat,0.95):.0f}ms")
print(f"rerank       int8-CPU: p50={pctl(int8_rerank_lat,0.5):.0f}ms  p95={pctl(int8_rerank_lat,0.95):.0f}ms")

combined_fp32_p95 = pctl(fp32_embed_lat, 0.95) + pctl(fp32_rerank_lat, 0.95)
combined_int8_p95 = pctl(int8_embed_lat, 0.95) + pctl(int8_rerank_lat, 0.95)
print(f"\ncombined embed+rerank p95 -- fp32: {combined_fp32_p95:.0f}ms, int8: {combined_int8_p95:.0f}ms "
      f"(target: under 1500ms; note this excludes Qdrant network time and sparse search, "
      f"measured separately since sparse stays on the original model)")

# these two fp32-CPU copies (~4.4GB combined) were only needed for this timing
# comparison -- free them before the recall benchmark's long loop below, which
# is the single biggest memory-pressure section in the whole notebook.
del cpu_embed_model, cpu_rerank_model
gc.collect()
print_mem("after latency benchmark, fp32-CPU models freed")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Chunks: 100%|██████████| 1/1 [00:00<00:00, 31.04it/s]


dense embed  fp32-CPU: p50=36ms  p95=134ms
dense embed  int8-CPU: p50=75ms  p95=111ms


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.69it/s]


rerank       fp32-CPU: p50=50ms  p95=59ms
rerank       int8-CPU: p50=137ms  p95=181ms

combined embed+rerank p95 -- fp32: 192ms, int8: 292ms (target: under 1500ms; note this excludes Qdrant network time and sparse search, measured separately since sparse stays on the original model)
[mem] after latency benchmark, fp32-CPU models freed: 12.38GB RSS, 0.00GB GPU


## Step 6: recall benchmark

Isolates exactly one variable -- fp32 vs int8 for dense embedding and
reranking -- against the same 248-row gold set. Sparse search and fusion
logic are identical in both arms (both use the original float32 sparse
output), and neither arm uses the conditional English leg or the
fusion-override guard from today's earlier fix, so this measures the ONNX
conversion's effect specifically, not anything else already validated.

In [12]:
from retrieval.pipeline import _get_retriever
from retrieval.search import HybridRetriever, COLLECTION, reciprocal_rank_fusion
from retrieval.rerank import rerank as original_rerank_fn

retriever_bench = _get_retriever()
print_mem("before recall benchmark (248 queries x 2 pipelines)")


def hybrid_embed_fn(text):
    # dense from onnx-int8, sparse from the original model -- the sparse head
    # isn't exportable via standard tooling, see the notebook intro above.
    dense_vec = onnx_dense_embed(text)
    sparse_vec = original_embed_text(text)["sparse"]
    return {"dense": dense_vec.tolist(), "sparse": sparse_vec}


int8_retriever = HybridRetriever(retriever_bench.client, collection=COLLECTION, embed_fn=hybrid_embed_fn)


def int8_rerank_fn(query, candidates, top_k=5):
    scored = []
    for c in candidates:
        row = dict(c)
        row["rerank_score"] = onnx_rerank_score(query, c["question"])
        scored.append(row)
    scored.sort(key=lambda r: -r["rerank_score"])
    return scored[:top_k]


def _fused_top1(retriever, rerank_fn, query, candidate_k=20):
    sd_dense = retriever.dense_search(query, top_k=candidate_k, lang="sd")
    sd_sparse = retriever.sparse_search(query, top_k=candidate_k, lang="sd")
    row_by_id = {}
    for rows in (sd_dense, sd_sparse):
        for row in rows:
            row_by_id.setdefault(row["answer_id"], row)
    fused = reciprocal_rank_fusion([
        [r["answer_id"] for r in sd_dense],
        [r["answer_id"] for r in sd_sparse],
    ])
    candidates = [row_by_id[aid] for aid, _ in fused[:candidate_k] if aid in row_by_id]
    reranked = rerank_fn(query, candidates, top_k=1) if candidates else []
    return reranked[0] if reranked else None


fp32_correct = 0
int8_correct = 0
for i, row in enumerate(gold_rows, start=1):
    gt_ids = row["_acceptable_ids"]
    top1_fp32 = _fused_top1(retriever_bench, original_rerank_fn, row["query"])
    top1_int8 = _fused_top1(int8_retriever, int8_rerank_fn, row["query"])
    if top1_fp32 and str(top1_fp32["answer_id"]) in gt_ids:
        fp32_correct += 1
    if top1_int8 and str(top1_int8["answer_id"]) in gt_ids:
        int8_correct += 1
    if i % 40 == 0:
        print(f"{i}/{len(gold_rows)}")

n = len(gold_rows)
recall_fp32 = fp32_correct / n
recall_int8 = int8_correct / n
drop_points = (recall_fp32 - recall_int8) * 100
print(f"\nRecall@1 fp32: {fp32_correct}/{n} = {recall_fp32:.3f}")
print(f"Recall@1 int8: {int8_correct}/{n} = {recall_int8:.3f}")
print(f"recall drop: {drop_points:.2f} percentage points (target: within 1.0 point of float32)")
print_mem("after recall benchmark")

[mem] before recall benchmark (248 queries x 2 pipelines): 12.39GB RSS, 0.00GB GPU


Chunks: 100%|██████████| 1/1 [00:00<00:00, 26.49it/s]


40/248


Chunks: 100%|██████████| 1/1 [00:00<00:00, 30.20it/s]


80/248


...

200/248


Chunks: 100%|██████████| 1/1 [00:00<00:00, 27.78it/s]


240/248


Chunks: 100%|██████████| 1/1 [00:00<00:00, 27.49it/s]



Recall@1 fp32: 87/248 = 0.351
Recall@1 int8: 81/248 = 0.327
recall drop: 2.42 percentage points (target: within 1.0 point of float32)
[mem] after recall benchmark: 12.40GB RSS, 0.00GB GPU


## Step 7: write results

In [13]:
import datetime

lines = []
lines.append("\n\n# Phase 2, Item 4 -- ONNX int8 conversion\n")
lines.append(f"Generated: {datetime.datetime.now(datetime.timezone.utc).isoformat()}, via retrieval/scripts/convert_models_to_onnx_int8.ipynb\n\n")
lines.append("**Scope:** converted and quantized the embedder's dense output and the reranker. "
              "Sparse search stays on the original float32 model -- its extra learned head isn't "
              "exportable via standard tooling, and converting it blind risked silently breaking "
              "a component that's currently pulling real weight (0.542 Recall@1 alone). "
              "See the notebook's intro cell for the full reasoning.\n\n")

lines.append("## Sanity checks (run before trusting anything below)\n")
lines.append(f"Reranker: relevant-pair / irrelevant-pair scores compared original vs int8 -- see notebook output.\n")
lines.append(f"Dense embedding: cosine similarity original vs onnx-int8 = {cos_sim:.4f} (target: >0.95, ideally >0.98).\n\n")

lines.append("## Latency (CPU, matching the free-Space deployment target)\n")
lines.append("| Stage | fp32-CPU p50 | fp32-CPU p95 | int8-CPU p50 | int8-CPU p95 |\n|---|---:|---:|---:|---:|\n")
lines.append(f"| Dense embed | {pctl(fp32_embed_lat,0.5):.0f}ms | {pctl(fp32_embed_lat,0.95):.0f}ms | {pctl(int8_embed_lat,0.5):.0f}ms | {pctl(int8_embed_lat,0.95):.0f}ms |\n")
lines.append(f"| Rerank | {pctl(fp32_rerank_lat,0.5):.0f}ms | {pctl(fp32_rerank_lat,0.95):.0f}ms | {pctl(int8_rerank_lat,0.5):.0f}ms | {pctl(int8_rerank_lat,0.95):.0f}ms |\n\n")
lines.append(f"Combined embed+rerank p95: fp32 {combined_fp32_p95:.0f}ms, int8 {combined_int8_p95:.0f}ms "
              f"(target: under 1500ms; excludes Qdrant network round-trip and sparse search).\n\n")

lines.append("## Recall (dense+rerank on int8, sparse and fusion unchanged, n=248)\n")
lines.append(f"Recall@1 fp32: {recall_fp32:.3f}. Recall@1 int8: {recall_int8:.3f}. "
              f"Drop: {drop_points:.2f} percentage points (target: within 1.0 point).\n\n")

verdict_latency = "PASS" if combined_int8_p95 < 1500 else "FAIL"
verdict_recall = "PASS" if abs(drop_points) <= 1.0 else "FAIL"
lines.append(f"**Verdict: latency {verdict_latency}, recall {verdict_recall}.**\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

appended to eval/results.md


# Phase 2, Item 4 -- ONNX int8 conversion
Generated: 2026-09-09T15:40:05.502119+00:00, via retrieval/scripts/convert_models_to_onnx_int8.ipynb

**Scope:** converted and quantized the embedder's dense output and the reranker. Sparse search stays on the original float32 model -- its extra learned head isn't exportable via standard tooling, and converting it blind risked silently breaking a component that's currently pulling real weight (0.542 Recall@1 alone). See the notebook's intro cell for the full reasoning.

## Sanity checks (run before trusting anything below)
Reranker: relevant-pair / irrelevant-pair scores compared original vs int8 -- see notebook output.
Dense embedding: cosine similarity original vs onnx-int8 = 0.9839 (target: >0.95, ideally >0.98).

## Latency (CPU, matching the free-Space deployment target)
| Stage | fp32-CPU p50 | fp32-CPU p95 | int8-CPU p50 | int8-CPU p95 |
|---|---:|---:|---:|---:|
| Dense embed | 36ms | 134ms | 75ms | 111ms |

In [14]:
import os

outputs = ["eval/results.md"]
outputs = [p for p in outputs if os.path.exists(p)]

try:
    from google.colab import files
    for p in outputs:
        files.download(p)
except ImportError:
    print("Not on Colab -- grab this from the notebook's file browser (Kaggle: the Output tab) instead:")
    for p in outputs:
        print(" ", os.path.abspath(p))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>